# 4b — Cell typing (chirp + STA clustering)

contact: ron.w.ditullio@gmail.com based on Guilhelm's guilhelm-dev branch

Clusters the good cells of the experiment into functional types from their **chirp PSTH**
and **STA time course**. It needs:

- notebook **4a** run with its chirp section (the cell quality file and the saved chirp
  rasters),
- notebook **2** (checkerboard STA) and notebook **3** (drifting gratings, for the DOS split).

Tested on Ubuntu 24.04.2 LTS (32 cores, 188 GiB RAM, Intel(R) Core(TM) i9-14900K)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import gc

import params
import utils
from utils import chirp as analysis

## Step 1: Load the cell quality and the chirp rasters

`selected_cells` = the cells that passed **every** criterion evaluated in 4a (RPV, STA and
chirp). To relax a criterion, e.g. cluster without the chirp review, list the criteria you
want: `utils.good_cells(cell_quality, criteria=["rpv_ok", "sta_ok"])`.

In [ ]:
cell_quality = utils.load_cell_quality(params)
selected_cells = utils.describe_cell_quality(cell_quality)  # cells passing every evaluated criterion

CT_directory = utils.find_analysis_directory(params.output_directory, "CellTyping")
check_directory = utils.find_analysis_directory(params.output_directory, "Checkerboard")
cell_data, old = analysis.load_chirp_rasters(CT_directory, params)

## Step 2: Select orientation-/direction-selective (DOS) cells

Uses the DG analysis (notebook 3). Cells that are **orientation-selective or direction-selective** — together called **DOS** — are split off from the rest, so the two groups are cell-typed separately below. Set `dos_cells` manually, or leave it empty to pick interactively from the DG plots (or reload a saved selection).

In [ ]:
# The DG analysis (notebook 3) must have been run for this experiment.
DG_directory = utils.find_analysis_directory(params.output_directory, "DG")

dos_cells = []  # set manually, e.g. [12, 45, 78] ; empty = interactive (or reload a saved selection)

dos_cells, non_dos_cells = utils.select_dos_cells(
    selected_cells, dos_cells, DG_directory, CT_directory, params
)

## Step 3: Cell typing — cluster non-DOS and DOS cells separately

Each group is clustered on its own with Agglomerative Clustering: first the non-DOS cells, then the DOS (orientation-/direction-selective) ones. The two groups can use different parameters.

-you want to move the distance threshold until you have roughly 20 clusters for non-DOS and 10 clusters for DOS <br>
-you want to have a number of PCs that cumulatively can explain around 80% of the variance in the dataset <br>
-you can decide how many components of the STA to choose in the clustering (usually 2 if the checkerboard recording is reliable and 1 otherwise)

In [ ]:
# --- Cluster the non-DOS cells ---
# Move dist_thres to adapt the dendrogram cut / number of clusters.
dist_thres_non_dos = 32  # Finetune to have ~20 non-DOS types
n_components_psth_non_dos = 16  # PCs from the chirp PSTH (aim for ~80% variance)
n_components_sta_tc_non_dos = (
    2  # PCs from the STA temporal course (2 if checkerboard reliable, else 1)
)
sparse = False


# run_cell_typing_AC drops cells with a flat/NaN PSTH or STA and returns the kept list,
# so non_dos_cells is updated to exactly the cells that were clustered.
psth_z_non_dos, sta_results, model_non_dos, non_dos_cells = utils.run_cell_typing_AC(
    dist_thres_non_dos,
    n_components_psth_non_dos,
    n_components_sta_tc_non_dos,
    cell_data,
    non_dos_cells,
    check_directory,
    sparse,
)

### Cluster the DOS (orientation-/direction-selective) cells

In [ ]:
# --- Cluster the DOS cells (their own parameters) ---
dist_thres_dos = 25  # Aim for ~10 types (less if you have few DOS cells \)
n_components_psth_dos = 8
n_components_sta_tc_dos = 2

# dos_cells is likewise updated to the cells actually clustered (flat/NaN ones dropped).
psth_z_dos, _, model_dos, dos_cells = utils.run_cell_typing_AC(
    dist_thres_dos,
    n_components_psth_dos,
    n_components_sta_tc_dos,
    cell_data,
    dos_cells,
    check_directory,
    sparse,
)

## Step 4: Merge the two labellings

DOS cluster IDs start right after the non-DOS ones, so the two never collide. Each clustered cell gets `cell_data[cell]["type"]` (cluster ID) and `cell_data[cell]["dos"]` (True/False — orientation- or direction-selective).

In [ ]:
# DOS cluster IDs start after the non-DOS ones so the two labellings don't collide.
# non_dos_cells / dos_cells are the cells that were actually clustered (run_cell_typing_AC
# may have dropped some); every other cell is left "Not assigned".
n_non_dos_clusters = len(np.unique(model_non_dos.labels_))

for i, cell in enumerate(non_dos_cells):
    cell_data[cell]["type"] = int(model_non_dos.labels_[i])
    cell_data[cell]["dos"] = False
for i, cell in enumerate(dos_cells):
    cell_data[cell]["type"] = int(model_dos.labels_[i]) + n_non_dos_clusters
    cell_data[cell]["dos"] = True

clustered = set(non_dos_cells) | set(dos_cells)
for cell in cell_data:
    if cell not in clustered:
        cell_data[cell]["type"] = "Not assigned"
        cell_data[cell]["dos"] = None

# Combined views for the downstream cells (cross-corr, summary): only the clustered
# cells, in cell_data order, with psth_z aligned to selected_cells.
selected_cells = [c for c in cell_data if c in clustered]
psth_z = np.vstack(
    [
        psth_z_non_dos[non_dos_cells.index(c)]
        if c in non_dos_cells
        else psth_z_dos[dos_cells.index(c)]
        for c in selected_cells
    ]
)

n_dos_clusters = len(np.unique(model_dos.labels_))
print(
    f"non-DOS clusters: {n_non_dos_clusters} | DOS clusters: {n_dos_clusters} "
    f"| total: {n_non_dos_clusters + n_dos_clusters} | clustered cells: {len(selected_cells)}"
)

## Step 5: Create a summary figure for each cluster type

In [ ]:
# selected_cells = the cells actually clustered (dropped cells excluded).
utils.create_cluster_summary_figure(
    cell_data, selected_cells, psth_z, sta_results, params, CT_directory, old
)

plt.close("all")

gc.collect()

## Step 6: When satisfied with the clustering, save data

In [ ]:
exp = params.exp
fsave = os.path.join(CT_directory, "{}_cell_typing_data".format(exp))
utils.save_obj(cell_data, fsave)

## (Optional) Plot a hand-made cluster

Group any cells you like into a named cluster and get the same mosaic figure. Set the
cluster name and the cell list below. The cells should be among the clustered cells
(`selected_cells`); the figure is saved as `Cluster_<name>.png` in the Cell_typing folder.

In [ ]:
handmade_cluster_name = ""  # name of the group (also the figure file name)
handmade_cell_list = []  # e.g. [208, 209, 210] ; the cells to group together

utils.plot_handmade_cluster(
    handmade_cluster_name,
    handmade_cell_list,
    cell_data,
    selected_cells,
    psth_z,
    sta_results,
    params,
    CT_directory,
    old,
)